In [1]:
import fitz
import pytesseract
from PIL import Image
import io
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

/home/muath/Documents/nezamy_rag/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def extract_text_from_pdf(pdf_path: str) -> str:
    """
    Extracts text from a PDF file using OCR (Tesseract).
    """
    print("Extracting text from PDF using OCR...")
    text_raw = ""
    with fitz.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf):
            pix = page.get_pixmap(matrix=fitz.Matrix(3, 3), alpha=False)
            image = Image.open(io.BytesIO(pix.tobytes("png")))
            page_text = pytesseract.image_to_string(image, lang="ara", config="--psm 6")
            text_raw += f"\n\n--- Page {page_number + 1} ---\n{page_text}"
    return text_raw

In [ ]:
def clean_arabic_text(raw_text: str) -> str:
    """
    Cleans the extracted Arabic text by removing noise, URLs, and extra spaces.
    """
    print("Cleaning the extracted text...")
    cleaned_data = []
    for line in raw_text.split('\n'):  
        line = line.strip()
        line = re.sub(r"\s+", " ", line)
        line = re.sub(r"http\S+|www\.\S+", "", line)
        line = re.sub(r"[^\u0600-\u06FFa-zA-Z0-9\s،؛؟.,:()\-]", "", line)
        line = re.sub(r"\s+", " ", line).strip()
        if line:
            cleaned_data.append(line)
    return "\n".join(cleaned_data)

In [ ]:
def hierarchical_legal_chunking(text: str) -> list:
    """
    Splits the legal text hierarchically (Chapter -> Section -> Article) 
    to preserve the legal context for each chunk.
    """
    print("Applying hierarchical chunking...")
    chunks = []
    current_chapter = ""
    current_section = ""
    current_article_text = ""

    lines = text.split('\n')
    for line in lines:
        line = line.strip()
        

        if not line or line.startswith("--- Page") or line.startswith("--- الصفحة"):
            continue
            

        if line.startswith("الباب"):
            current_chapter = line
            current_section = "" 
            continue
            

        elif line.startswith("الفصل"):
            current_section = line
            continue
            

        elif line.startswith("المادة"):
            if current_article_text:
                chunks.append(current_article_text.strip())

            context_header = f"[{current_chapter}"
            if current_section:
                context_header += f" - {current_section}"
            context_header += "]\n"
            
            current_article_text = context_header + line + "\n"
            

        else:
            if current_article_text: 
                current_article_text += line + "\n"
                
    if current_article_text:
        chunks.append(current_article_text.strip())
        
    return chunks

In [ ]:
def split_and_build_db(chunks: list, db_path: str):
    """
    Applies a secondary split (max 1500 chars) for safety and builds the Chroma Vector DB.
    """
    print("Applying secondary split (RecursiveCharacterTextSplitter)...")
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1500,      
        chunk_overlap=0,    
        separators=["\n\n", "\n", " ", ""]
    )
    final_documents = text_splitter.create_documents(chunks)
    print(f"Prepared {len(final_documents)} final document chunks.")

    print("Building Chroma Vector Database...")
    embedding_model = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-base")
    Chroma.from_documents(
        documents=final_documents,
        embedding=embedding_model,
        persist_directory=db_path
    )
    print("Vector Database successfully built and saved!")